# 1.3 金融市场基础

## 学习目标
- 了解主要金融资产类别：股票、ETF、期货、期权
- 理解市场微观结构：订单类型、买卖价差
- 熟悉交易成本的影响

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (12, 5)

## 1. 主要资产类别

| 资产 | 说明 | 特点 | 示例 |
|------|------|------|------|
| **股票 (Stock)** | 公司股权凭证 | 高风险高收益，日内可多次交易 | AAPL, 贵州茅台 |
| **ETF** | 跟踪指数的基金 | 分散化、低费率、T+0（美股） | SPY, 510050 |
| **期货 (Futures)** | 约定未来买卖的合约 | 有杠杆、有到期日、可做空 | ES, IF |
| **期权 (Options)** | 买卖权利而非义务 | 非线性收益、灵活对冲 | SPY Put/Call |
| **债券 (Bonds)** | 借贷凭证 | 固定利息、低风险 | 国债, TLT |
| **外汇 (FX)** | 货币对交易 | 24小时、高流动性 | EURUSD |
| **加密货币** | 去中心化数字货币 | 极高波动率、7×24h | BTC, ETH |

## 2. 股票 K 线图解读

In [ ]:
import yfinance as yf

# 下载近30个交易日数据
df = yf.download('AAPL', period='60d', progress=False)
df = df.tail(30).copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), 
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# K线图
for idx, (date, row) in enumerate(df.iterrows()):
    o, h, l, c = float(row['Open']), float(row['High']), float(row['Low']), float(row['Close'])
    color = 'green' if c >= o else 'red'
    # 实体
    rect = mpatches.FancyBboxPatch((idx - 0.3, min(o, c)), 0.6, abs(c - o),
                                    color=color, alpha=0.8)
    ax1.add_patch(rect)
    # 上下影线
    ax1.plot([idx, idx], [l, min(o, c)], color=color, linewidth=1)
    ax1.plot([idx, idx], [max(o, c), h], color=color, linewidth=1)

ax1.set_xlim(-1, len(df))
ax1.set_ylim(df['Low'].min().item() * 0.99, df['High'].max().item() * 1.01)
ax1.set_title('AAPL K线图 (近30交易日)', fontsize=13)
ax1.set_ylabel('价格 (USD)')
ax1.grid(alpha=0.2)

# 成交量
for idx, (date, row) in enumerate(df.iterrows()):
    o, c = float(row['Open']), float(row['Close'])
    color = 'green' if c >= o else 'red'
    ax2.bar(idx, float(row['Volume']) / 1e6, color=color, alpha=0.7, width=0.8)

ax2.set_ylabel('成交量 (M股)')
ax2.grid(alpha=0.2)

# X轴日期标签
step = max(1, len(df) // 8)
ax2.set_xticks(range(0, len(df), step))
ax2.set_xticklabels([df.index[i].strftime('%m-%d') for i in range(0, len(df), step)],
                     rotation=30)
plt.tight_layout()
plt.show()
print('绿色=阳线（收>开），红色=阴线（收<开）')

## 3. 订单类型

| 订单类型 | 说明 | 优点 | 缺点 |
|----------|------|------|------|
| **市价单 (Market Order)** | 以当前最优价格立即成交 | 保证成交 | 价格不确定（滑点） |
| **限价单 (Limit Order)** | 指定价格或更好价格成交 | 控制价格 | 可能不成交 |
| **止损单 (Stop Order)** | 触发价格后转为市价单 | 控制亏损 | 波动时价格可能滑点大 |
| **止损限价单** | 触发后转为限价单 | 两者结合 | 可能无法成交 |

## 4. 交易成本的影响

In [ ]:
# 模拟交易成本对策略的影响
np.random.seed(42)
n_trades = 200
# 假设每笔交易有 55% 的胜率，盈亏比 1:1
win_rate = 0.55
win_amount = 0.01   # 每次盈利 1%
loss_amount = -0.01  # 每次亏损 1%

outcomes = np.random.choice([win_amount, loss_amount], 
                              size=n_trades, p=[win_rate, 1 - win_rate])

# 不同手续费率
commission_rates = [0, 0.001, 0.002, 0.005]
labels = ['无成本', '0.1%', '0.2%（常见）', '0.5%']

fig, ax = plt.subplots(figsize=(12, 5))
for rate, label in zip(commission_rates, labels):
    net = outcomes - rate  # 每笔扣除手续费
    cum = (1 + net).cumprod() * 10000
    ax.plot(cum, label=label, linewidth=1.5)

ax.axhline(10000, color='gray', linestyle='--', alpha=0.5)
ax.set_title(f'200笔交易 × 胜率{win_rate:.0%}：交易成本对收益的侵蚀', fontsize=13)
ax.set_xlabel('交易次数')
ax.set_ylabel('资产价值')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('结论：高频交易中，手续费是策略的关键变量！')

## 🎯 练习

1. 观察 K 线图中的**长上影线**和**长下影线**分别意味着什么？
2. 将上方模拟中 `win_rate` 改为 0.50，观察不同手续费下的结果。当没有成本时，50% 胜率能赚钱吗？
3. 期权最大亏损是什么？查阅资料后在下方用 Markdown 写下你的理解。

---
**下一模块** → `../02_data/01_data_sources.ipynb`